# 03.2 Isolation Forest (REVISED)

**Вход (артефакты из 03.1):**
- `split_metadata.json` — train_core / calibration_val / test flight ids
- `clip_bounds.json` — P0.1/P99.9 границы 12 model features
- `scaler.joblib` — StandardScaler
- `contract.json` — model features, DQ-флаги, clean policy

**Выход:**
- `if_main.joblib`, `if_seed_2.joblib`, `if_seed_3.joblib` — модели
- `if_scores.parquet` — point-level scores для calibration_val + test
- `if_stability.json` — Jaccard / Pearson stability metrics

---

## Изменения vs первая версия 03.2

После критического review:

1. **Percentile reference на CLEAN calibration_val** (не all). DQ-tainted
   точки calibration_val не должны определять шкалу anomaly score.
2. **Streaming Pass A → Pass B** через temp parquet chunks вместо
   `pd.concat(scored_chunks)` в RAM. Безопасно для 63M scored points.
3. **`row_id`** добавлен — глобальный sequential index по annotated_v3,
   будет use'ится в 03.5 для O(n) merge с HDBSCAN/LSTM scores.
4. **NaN fallback** для редких phase percentiles → global percentile.
5. **DQ flags читаются в scoring и сохраняются в output** — для удобства
   downstream и для построения clean reference.
6. **`np.array(sorted(set))` вместо `list(set)`** — ускоряет `np.isin`.
7. **Pre-fill phase reservoirs batch-wise** — быстрее loop'а pre-fill.
8. **Stability на all+clean** — две метрики для лучшей интерпретации.

## Архитектура трёх passes

```
Pass A (scoring + reference accumulation):
  for batch in iter_batches:
      score eligible points → temp chunk parquet
      accumulate clean calibration scores per phase in RAM (light)

Pass B (percentile + streaming write):
  sort references in RAM
  for chunk in temp_chunks:
      compute global_pct + phase_pct via ECDF
      write to final if_scores.parquet via ParquetWriter
  cleanup temp chunks

Pass C (stability):
  read calibration_val section of final parquet
  compute Jaccard / Pearson on all + clean subsets
```

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import os
import gc
import json
import time
import glob
from sklearn.ensemble import IsolationForest
import joblib

DATA_DIR = '/content/drive/MyDrive/thesis_processed'
ARTIFACTS_DIR = os.path.join(DATA_DIR, 'models_v3_artifacts')

INPUT_PATH = os.path.join(DATA_DIR, 'european_flights_annotated_v3.parquet')

# Read artifacts from 03.1
SPLIT_PATH = os.path.join(ARTIFACTS_DIR, 'split_metadata.json')
CLIP_BOUNDS_PATH = os.path.join(ARTIFACTS_DIR, 'clip_bounds.json')
SCALER_PATH = os.path.join(ARTIFACTS_DIR, 'scaler.joblib')
CONTRACT_PATH = os.path.join(ARTIFACTS_DIR, 'contract.json')

# Output artifacts (03.2)
IF_MAIN_PATH = os.path.join(ARTIFACTS_DIR, 'if_main.joblib')
IF_SEED2_PATH = os.path.join(ARTIFACTS_DIR, 'if_seed_2.joblib')
IF_SEED3_PATH = os.path.join(ARTIFACTS_DIR, 'if_seed_3.joblib')
IF_SCORES_PATH = os.path.join(ARTIFACTS_DIR, 'if_scores.parquet')
IF_STABILITY_PATH = os.path.join(ARTIFACTS_DIR, 'if_stability.json')

# Temp directory for Pass A chunks
TEMP_DIR = os.path.join(ARTIFACTS_DIR, 'if_temp_chunks')

print(f'Input:     {INPUT_PATH}')
print(f'Artifacts: {ARTIFACTS_DIR}')
print(f'Inputs exist:')
for path in [INPUT_PATH, SPLIT_PATH, CLIP_BOUNDS_PATH, SCALER_PATH, CONTRACT_PATH]:
    print(f'  {os.path.basename(path):>22s}: {os.path.exists(path)}')

Mounted at /content/drive
Input:     /content/drive/MyDrive/thesis_processed/european_flights_annotated_v3.parquet
Artifacts: /content/drive/MyDrive/thesis_processed/models_v3_artifacts
Inputs exist:
  european_flights_annotated_v3.parquet: True
     split_metadata.json: True
        clip_bounds.json: True
           scaler.joblib: True
           contract.json: True


## 2. Load artifacts from 03.1

In [2]:
with open(CONTRACT_PATH) as f:
    contract = json.load(f)

MODEL_FEATURES = contract['model_features']
DQ_HARD = contract['dq_hard_flag']
DQ_SOFT = contract['dq_soft_flag']
FEATURE_QUALITY = contract['feature_quality_flag']
RANDOM_STATE = contract['random_state']

with open(SPLIT_PATH) as f:
    split_metadata = json.load(f)

# Pre-allocated sorted int64 arrays (быстрее чем list(set) per-batch)
train_core_ids = np.array(sorted(split_metadata['train_core_flights']), dtype=np.int64)
calibration_val_ids = np.array(sorted(split_metadata['calibration_val_flights']), dtype=np.int64)
test_ids = np.array(sorted(split_metadata['test_flights']), dtype=np.int64)

print(f'Splits:')
print(f'  train_core:      {len(train_core_ids):>6,} flights')
print(f'  calibration_val: {len(calibration_val_ids):>6,} flights')
print(f'  test:            {len(test_ids):>6,} flights')

with open(CLIP_BOUNDS_PATH) as f:
    clip_data = json.load(f)
clip_bounds = clip_data['bounds']

scaler = joblib.load(SCALER_PATH)
print(f'\nScaler loaded: mean shape {scaler.mean_.shape}')

Splits:
  train_core:      17,230 flights
  calibration_val:  4,307 flights
  test:             8,251 flights

Scaler loaded: mean shape (12,)


In [3]:
# IF parameters
IF_N_ESTIMATORS = 300
IF_MAX_SAMPLES = 2048
IF_POINTS_PER_PHASE = 100_000

IF_SEED_MAIN = RANDOM_STATE          # 1321
IF_SEED_2 = RANDOM_STATE + 1000      # 2321
IF_SEED_3 = RANDOM_STATE + 2000      # 3321

print(f'\nIF parameters:')
print(f'  n_estimators:      {IF_N_ESTIMATORS}')
print(f'  max_samples:       {IF_MAX_SAMPLES}')
print(f'  points_per_phase:  {IF_POINTS_PER_PHASE:,}')
print(f'  seeds: main={IF_SEED_MAIN}, stab2={IF_SEED_2}, stab3={IF_SEED_3}')


IF parameters:
  n_estimators:      300
  max_samples:       2048
  points_per_phase:  100,000
  seeds: main=1321, stab2=2321, stab3=3321


## 3. Helper: clip + transform features

In [4]:
def clip_and_scale(X_raw):
    """X_raw: (n, 12) float array, raw model features.
    Returns scaled (n, 12) float32 array — clip + StandardScaler.
    """
    X = X_raw.astype(np.float32, copy=True)
    for i, feature in enumerate(MODEL_FEATURES):
        bounds = clip_bounds[feature]
        np.clip(X[:, i], bounds['low'], bounds['high'], out=X[:, i])
    X_scaled = scaler.transform(X)
    return X_scaled.astype(np.float32)


print('clip_and_scale defined.')

clip_and_scale defined.


## 4. Build phase-balanced training sample (vectorized pre-fill + reservoir replace)

Single streaming pass через annotated_v3:

1. Phase reservoirs накапливаются с pre-fill пока размер < 100K (быстро,
   просто vstack).
2. После заполнения — switch на reservoir replace через rng.randint
   (медленнее, но per-replace O(1)).

Это даёт несколько раз faster pre-fill phase для частых фаз.

In [5]:
print('Building phase-balanced clean train_core sample...')
t0 = time.time()

PHASE_NAMES = {0: 'ground', 1: 'takeoff', 2: 'climb', 3: 'cruise',
               4: 'descent', 5: 'approach', 6: 'landing'}
PHASE_CODES = list(PHASE_NAMES.keys())  # [0..6], исключаем -1 unknown

phase_reservoirs = {p: [] for p in PHASE_CODES}  # list of np arrays
phase_reservoir_sizes = {p: 0 for p in PHASE_CODES}  # current size
phase_seen = {p: 0 for p in PHASE_CODES}  # total seen

read_cols = ['flight_id', 'flight_phase', DQ_HARD, DQ_SOFT, FEATURE_QUALITY] + MODEL_FEATURES
batch_size = 5_000_000
pf = pq.ParquetFile(INPUT_PATH)

rng = np.random.RandomState(IF_SEED_MAIN)

n_batches = 0
for batch in pf.iter_batches(batch_size=batch_size, columns=read_cols):
    df_batch = batch.to_pandas()
    n_batches += 1

    fid = df_batch['flight_id'].to_numpy()
    is_train_core = np.isin(fid, train_core_ids)

    finite_mask = df_batch[MODEL_FEATURES].notna().all(axis=1).to_numpy()
    clean_mask = (
        finite_mask
        & is_train_core
        & (df_batch[DQ_HARD].to_numpy() == 0)
        & (df_batch[DQ_SOFT].to_numpy() == 0)
        & (df_batch[FEATURE_QUALITY].to_numpy() == 0)
    )

    if not clean_mask.any():
        del df_batch, fid, is_train_core, finite_mask, clean_mask
        gc.collect()
        continue

    phases_in_batch = df_batch.loc[clean_mask, 'flight_phase'].to_numpy()
    features_in_batch = df_batch.loc[clean_mask, MODEL_FEATURES].to_numpy(dtype=np.float32)

    for phase in PHASE_CODES:
        phase_mask_in_clean = phases_in_batch == phase
        if not phase_mask_in_clean.any():
            continue
        phase_features = features_in_batch[phase_mask_in_clean]
        n_in_phase = len(phase_features)
        phase_seen[phase] += n_in_phase

        current_size = phase_reservoir_sizes[phase]

        if current_size < IF_POINTS_PER_PHASE:
            # Pre-fill phase: vstack
            n_to_add = min(n_in_phase, IF_POINTS_PER_PHASE - current_size)
            phase_reservoirs[phase].append(phase_features[:n_to_add])
            phase_reservoir_sizes[phase] += n_to_add

            # Если есть точки сверх pre-fill — reservoir replace
            if n_to_add < n_in_phase:
                remaining = phase_features[n_to_add:]
                seen_offset = phase_seen[phase] - n_in_phase + n_to_add
                if len(phase_reservoirs[phase]) > 1:
                    phase_reservoirs[phase] = [np.vstack(phase_reservoirs[phase])]
                reservoir_arr = phase_reservoirs[phase][0]
                for i, point in enumerate(remaining):
                    j = rng.randint(0, seen_offset + i + 1)
                    if j < IF_POINTS_PER_PHASE:
                        reservoir_arr[j] = point
        else:
            # Replace phase
            if len(phase_reservoirs[phase]) > 1:
                phase_reservoirs[phase] = [np.vstack(phase_reservoirs[phase])]
            reservoir_arr = phase_reservoirs[phase][0]
            seen_offset = phase_seen[phase] - n_in_phase
            for i, point in enumerate(phase_features):
                j = rng.randint(0, seen_offset + i + 1)
                if j < IF_POINTS_PER_PHASE:
                    reservoir_arr[j] = point

    sizes_str = ', '.join(f'{PHASE_NAMES[p][:3]}={phase_reservoir_sizes[p]}'
                          for p in PHASE_CODES)
    print(f'  batch {n_batches}: {sizes_str}', end='\r')

    del df_batch, fid, is_train_core, finite_mask, clean_mask
    del phases_in_batch, features_in_batch
    gc.collect()

print()
print(f'\nReservoir sampling complete: {time.time() - t0:.0f}s')
print(f'Per-phase counts:')
for phase in PHASE_CODES:
    seen = phase_seen[phase]
    kept = phase_reservoir_sizes[phase]
    print(f'  {PHASE_NAMES[phase]:>10s}: kept {kept:>7,} of {seen:>10,} seen')

# Combine reservoirs
X_train_list = []
phase_train_list = []
for phase in PHASE_CODES:
    if phase_reservoir_sizes[phase] > 0:
        if len(phase_reservoirs[phase]) > 1:
            arr = np.vstack(phase_reservoirs[phase])
        else:
            arr = phase_reservoirs[phase][0]
        X_train_list.append(arr)
        phase_train_list.append(np.full(len(arr), phase, dtype=np.int8))

X_train_raw = np.vstack(X_train_list)
phase_train = np.concatenate(phase_train_list)

del phase_reservoirs, X_train_list, phase_train_list
gc.collect()

print(f'\nTotal training points: {len(X_train_raw):,}')
print(f'Training matrix memory: {X_train_raw.nbytes / 1e6:.1f} MB')

# Apply clip + scaler
X_train = clip_and_scale(X_train_raw)
print(f'After clip + scale: {X_train.shape}, dtype {X_train.dtype}')
del X_train_raw
gc.collect()

Building phase-balanced clean train_core sample...


Reservoir sampling complete: 360s
Per-phase counts:
      ground: kept 100,000 of    190,763 seen
     takeoff: kept 100,000 of    173,361 seen
       climb: kept 100,000 of 15,908,144 seen
      cruise: kept 100,000 of 43,765,591 seen
     descent: kept 100,000 of 14,367,860 seen
    approach: kept 100,000 of  9,155,285 seen
     landing: kept 100,000 of    821,913 seen

Total training points: 700,000
Training matrix memory: 33.6 MB
After clip + scale: (700000, 12), dtype float32


0

## 5. Train Isolation Forest (3 seeds for stability)

In [6]:
print('Training Isolation Forest (3 seeds for stability)...')

models = {}
for seed_name, seed in [('main', IF_SEED_MAIN), ('seed_2', IF_SEED_2), ('seed_3', IF_SEED_3)]:
    t0 = time.time()
    print(f'\n  Training {seed_name} (random_state={seed})...')
    iforest = IsolationForest(
        n_estimators=IF_N_ESTIMATORS,
        max_samples=IF_MAX_SAMPLES,
        contamination='auto',
        random_state=seed,
        n_jobs=-1,
        bootstrap=False,
    )
    iforest.fit(X_train)
    models[seed_name] = iforest
    print(f'    Time: {time.time() - t0:.0f}s')

joblib.dump(models['main'], IF_MAIN_PATH)
joblib.dump(models['seed_2'], IF_SEED2_PATH)
joblib.dump(models['seed_3'], IF_SEED3_PATH)
print(f'\nSaved 3 models.')

del X_train
gc.collect()

Training Isolation Forest (3 seeds for stability)...

  Training main (random_state=1321)...
    Time: 4s

  Training seed_2 (random_state=2321)...
    Time: 4s

  Training seed_3 (random_state=3321)...
    Time: 3s

Saved 3 models.


24

## 6. Pass A — score + write temp chunks + accumulate clean reference

Streaming через annotated_v3:
- Score all eligible (calibration_val ∪ test ∩ finite features) точек.
- Стримим raw scores в temp chunk parquets.
- В RAM накапливаем **только clean calibration scores** per phase
  (для percentile reference в Pass B).

`row_id` назначается как глобальный sequential index по всем строкам
annotated_v3 (включая non-eligible) — это обеспечивает консистентность
с любыми другими scoring passes (HDBSCAN, LSTM в 03.3-03.4).

In [7]:
print('Pass A: scoring + accumulating clean calibration reference...')
t0 = time.time()

# Cleanup old temp chunks
os.makedirs(TEMP_DIR, exist_ok=True)
old_temp = glob.glob(os.path.join(TEMP_DIR, '*.parquet'))
if old_temp:
    for old in old_temp:
        os.remove(old)
    print(f'Removed {len(old_temp)} stale temp chunks.')

# Reference accumulators (in RAM)
calib_clean_global_scores = []  # list of np arrays
calib_clean_phase_scores = {p: [] for p in PHASE_CODES + [-1]}

read_cols_score = ['flight_id', 'timestamp', 'flight_phase',
                   DQ_HARD, DQ_SOFT, FEATURE_QUALITY] + MODEL_FEATURES

global_offset = 0
n_batches = 0
n_scored = 0
chunk_idx = 0

for batch in pf.iter_batches(batch_size=batch_size, columns=read_cols_score):
    df_batch = batch.to_pandas()
    n = len(df_batch)
    n_batches += 1

    fid = df_batch['flight_id'].to_numpy()
    is_calval = np.isin(fid, calibration_val_ids)
    is_test = np.isin(fid, test_ids)
    is_eligible = is_calval | is_test

    if not is_eligible.any():
        global_offset += n
        del df_batch, fid, is_calval, is_test, is_eligible
        gc.collect()
        continue

    finite_mask = df_batch[MODEL_FEATURES].notna().all(axis=1).to_numpy()
    score_mask = is_eligible & finite_mask

    if not score_mask.any():
        global_offset += n
        del df_batch, fid, is_calval, is_test, is_eligible, finite_mask, score_mask
        gc.collect()
        continue

    # row_id для score_mask points
    row_ids = np.arange(global_offset, global_offset + n, dtype=np.int64)[score_mask]

    # Scoring inputs
    X_raw = df_batch.loc[score_mask, MODEL_FEATURES].to_numpy()
    X = clip_and_scale(X_raw)

    # Main scores
    scores_main = (-models['main'].score_samples(X)).astype(np.float32)

    # Stability scores только для calval points
    is_calval_in_score = is_calval[score_mask]
    n_score = score_mask.sum()
    scores_s2 = np.full(n_score, np.nan, dtype=np.float32)
    scores_s3 = np.full(n_score, np.nan, dtype=np.float32)
    if is_calval_in_score.any():
        X_calval = X[is_calval_in_score]
        scores_s2[is_calval_in_score] = (
            -models['seed_2'].score_samples(X_calval)).astype(np.float32)
        scores_s3[is_calval_in_score] = (
            -models['seed_3'].score_samples(X_calval)).astype(np.float32)

    # DQ flags для всех scored points
    dq_hard_arr = df_batch.loc[score_mask, DQ_HARD].to_numpy().astype('uint8')
    dq_soft_arr = df_batch.loc[score_mask, DQ_SOFT].to_numpy().astype('uint8')
    fq_arr = df_batch.loc[score_mask, FEATURE_QUALITY].to_numpy().astype('uint8')
    is_clean_ref = ((dq_hard_arr == 0) & (dq_soft_arr == 0) & (fq_arr == 0)).astype('uint8')

    phases_arr = df_batch.loc[score_mask, 'flight_phase'].to_numpy().astype('int8')

    # Build chunk DataFrame
    chunk = pd.DataFrame({
        'row_id': row_ids,
        'flight_id': df_batch.loc[score_mask, 'flight_id'].to_numpy().astype('int64'),
        'timestamp': df_batch.loc[score_mask, 'timestamp'].to_numpy(),
        'flight_phase': phases_arr,
        'split': pd.Categorical(
            np.where(is_calval_in_score, 'calibration_val', 'test'),
            categories=['calibration_val', 'test']
        ),
        'dq_hard_flag': dq_hard_arr,
        'dq_soft_flag': dq_soft_arr,
        'feature_quality_flag': fq_arr,
        'is_clean_reference': is_clean_ref,
        'if_score_raw_main': scores_main,
        'if_score_raw_s2': scores_s2,
        'if_score_raw_s3': scores_s3,
    })

    chunk_path = os.path.join(TEMP_DIR, f'if_chunk_{chunk_idx:03d}.parquet')
    chunk.to_parquet(chunk_path, index=False, compression='snappy')
    chunk_idx += 1

    # Накопление clean calibration reference
    cal_clean_score_mask = is_calval_in_score & (is_clean_ref == 1)
    if cal_clean_score_mask.any():
        cal_clean_scores = scores_main[cal_clean_score_mask]
        cal_clean_phases = phases_arr[cal_clean_score_mask]
        calib_clean_global_scores.append(cal_clean_scores)
        for p in np.unique(cal_clean_phases):
            mask_p = cal_clean_phases == p
            calib_clean_phase_scores[int(p)].append(cal_clean_scores[mask_p])

    n_scored += n_score
    global_offset += n

    print(f'  batch {n_batches}: scored = {n_scored:,}, temp chunks = {chunk_idx}',
          end='\r')

    del df_batch, fid, is_calval, is_test, is_eligible
    del finite_mask, score_mask, X_raw, X
    del scores_main, scores_s2, scores_s3
    del dq_hard_arr, dq_soft_arr, fq_arr, is_clean_ref, phases_arr
    del chunk
    gc.collect()

print()
print(f'\nPass A complete: {time.time() - t0:.0f}s')
print(f'Total scored points: {n_scored:,}')
print(f'Temp chunks written: {chunk_idx}')

# Combine + sort references
print('\nBuilding clean calibration reference distributions...')
ref_global_arr = np.concatenate(calib_clean_global_scores)
ref_global_sorted = np.sort(ref_global_arr)
n_ref_global = len(ref_global_sorted)
print(f'  Global reference (clean calibration_val): {n_ref_global:,} points')
print(f'  Range: [{ref_global_sorted[0]:.4f}, {ref_global_sorted[-1]:.4f}]')
print(f'  P50/P90/P99/P99.9: '
      f'{np.quantile(ref_global_sorted, [0.5, 0.9, 0.99, 0.999]).round(4)}')

ref_phase_sorted = {}
for p in PHASE_CODES + [-1]:
    if calib_clean_phase_scores[p]:
        arr = np.concatenate(calib_clean_phase_scores[p])
        if len(arr) >= 100:
            ref_phase_sorted[p] = np.sort(arr)
            name = PHASE_NAMES.get(p, 'unknown')
            print(f'  {name:>10s}: ref n={len(arr):>10,}')
        else:
            name = PHASE_NAMES.get(p, 'unknown')
            print(f'  WARN {name}: only {len(arr)} clean cal points — '
                  f'phase pct will fall back to global')

del calib_clean_global_scores, calib_clean_phase_scores, ref_global_arr
gc.collect()

Pass A: scoring + accumulating clean calibration reference...
  batch 30: scored = 62,578,761, temp chunks = 30

Pass A complete: 1817s
Total scored points: 62,578,761
Temp chunks written: 30

Building clean calibration reference distributions...
  Global reference (clean calibration_val): 21,083,531 points
  Range: [0.3464, 0.7101]
  P50/P90/P99/P99.9: [0.4061 0.4668 0.552  0.6069]
      ground: ref n=    47,763
     takeoff: ref n=    41,899
       climb: ref n= 3,922,711
      cruise: ref n=10,987,260
     descent: ref n= 3,572,649
    approach: ref n= 2,306,701
     landing: ref n=   204,548


0

## 7. Pass B — read chunks, compute percentiles, write final parquet

Streaming через temp chunks:
- Apply ECDF for global_pct against clean calibration reference.
- Apply per-phase ECDF, fallback to global if phase reference < 100.
- Write to final `if_scores.parquet` via ParquetWriter.

После Pass B temp chunks удаляются.

In [9]:
print('Pass B: applying percentile ECDF and writing final parquet...')
t0 = time.time()

temp_chunks = sorted(glob.glob(os.path.join(TEMP_DIR, '*.parquet')))
print(f'Reading {len(temp_chunks)} temp chunks...')

writer = None
total_written = 0
n_phase_fallback = 0

for cp in temp_chunks:
    chunk = pd.read_parquet(cp)

    # Global pct
    chunk['if_score_global_pct'] = (
        np.searchsorted(
            ref_global_sorted, chunk['if_score_raw_main'].to_numpy(),
            side='right'
        ) / n_ref_global
    ).astype('float32')

    # Phase pct
    phase_pct = np.full(len(chunk), np.nan, dtype=np.float32)
    phases = chunk['flight_phase'].to_numpy()
    for p, ref_p in ref_phase_sorted.items():
        mask = phases == p
        if not mask.any():
            continue
        phase_pct[mask] = (
            np.searchsorted(ref_p, chunk.loc[mask, 'if_score_raw_main'].to_numpy(),
                            side='right') / len(ref_p)
        )
    chunk['if_score_phase_pct'] = phase_pct

    # Fallback NaN → global
    nan_mask = np.isnan(phase_pct)
    n_phase_fallback += int(nan_mask.sum())
    if nan_mask.any():
        chunk.loc[nan_mask, 'if_score_phase_pct'] = chunk.loc[
            nan_mask, 'if_score_global_pct'
        ]
    chunk['if_score_phase_pct'] = chunk['if_score_phase_pct'].astype('float32')

    # Write
    table = pa.Table.from_pandas(chunk, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(IF_SCORES_PATH, table.schema, compression='snappy')
    writer.write_table(table)
    total_written += len(chunk)

    print(f'  {os.path.basename(cp)}: {len(chunk):,} rows', end='\r')

    del chunk, table
    gc.collect()

if writer is not None:
    writer.close()

print()
print(f'\nPass B complete: {time.time() - t0:.0f}s')
print(f'Total written to {IF_SCORES_PATH}: {total_written:,}')
print(f'Phase pct fallback to global: {n_phase_fallback:,} '
      f'({n_phase_fallback / total_written * 100:.4f}%)')
print(f'File size: {os.path.getsize(IF_SCORES_PATH) / 1e6:.1f} MB')

Pass B: applying percentile ECDF and writing final parquet...
Reading 30 temp chunks...
  if_chunk_029.parquet: 4,094,536 rows

Pass B complete: 109s
Total written to /content/drive/MyDrive/thesis_processed/models_v3_artifacts/if_scores.parquet: 62,578,761
Phase pct fallback to global: 0 (0.0000%)
File size: 1363.4 MB


In [10]:
# Cleanup temp chunks
for cp in temp_chunks:
    os.remove(cp)
try:
    os.rmdir(TEMP_DIR)
    print(f'Removed temp directory and {len(temp_chunks)} chunks.')
except OSError:
    print(f'Removed {len(temp_chunks)} temp chunks (dir kept).')

del ref_global_sorted, ref_phase_sorted
gc.collect()

Removed temp directory and 30 chunks.


0

## 8. Pass C — Stability analysis (all + clean)

Reads calibration_val section of final parquet. Computes Jaccard top-1%
between all 3 seed scores, on:
- **All calibration_val** points (full mix including DQ-tainted).
- **Clean calibration_val only** (`is_clean_reference == 1`).

Clean stability — более честная оценка модели как detector нормальных
аномалий. All — полная reproducibility.

In [11]:
print('Pass C: stability analysis on calibration_val...')

# Read только calval points с seed scores
calval = pq.read_table(
    IF_SCORES_PATH,
    columns=['split', 'is_clean_reference',
             'if_score_raw_main', 'if_score_raw_s2', 'if_score_raw_s3'],
    filters=[('split', '=', 'calibration_val')]
).to_pandas()

calval = calval.dropna(subset=['if_score_raw_s2', 'if_score_raw_s3'])
n_all = len(calval)
n_clean = int((calval['is_clean_reference'] == 1).sum())

print(f'\nCalibration_val with all 3 seed scores: {n_all:,}')
print(f'  Clean subset: {n_clean:,} ({n_clean / n_all * 100:.2f}%)')

Pass C: stability analysis on calibration_val...

Calibration_val with all 3 seed scores: 21,390,254
  Clean subset: 21,083,531 (98.57%)


In [12]:
def jaccard_top_pct(scores_a, scores_b, top_pct=0.01):
    """Jaccard between top-N points by two score arrays."""
    n = len(scores_a)
    n_top = max(1, int(n * top_pct))
    top_a = set(np.argsort(scores_a)[-n_top:])
    top_b = set(np.argsort(scores_b)[-n_top:])
    inter = len(top_a & top_b)
    union = len(top_a | top_b)
    return inter / union if union > 0 else 0.0


def compute_stability_metrics(s_main, s_2, s_3):
    return {
        'n_points': len(s_main),
        'jaccard_top_1pct': {
            'main_vs_s2': float(jaccard_top_pct(s_main, s_2, 0.01)),
            'main_vs_s3': float(jaccard_top_pct(s_main, s_3, 0.01)),
            's2_vs_s3': float(jaccard_top_pct(s_2, s_3, 0.01)),
        },
        'jaccard_top_0_5pct': {
            'main_vs_s2': float(jaccard_top_pct(s_main, s_2, 0.005)),
            'main_vs_s3': float(jaccard_top_pct(s_main, s_3, 0.005)),
            's2_vs_s3': float(jaccard_top_pct(s_2, s_3, 0.005)),
        },
        'jaccard_top_0_1pct': {
            'main_vs_s2': float(jaccard_top_pct(s_main, s_2, 0.001)),
            'main_vs_s3': float(jaccard_top_pct(s_main, s_3, 0.001)),
            's2_vs_s3': float(jaccard_top_pct(s_2, s_3, 0.001)),
        },
        'pearson': {
            'main_vs_s2': float(np.corrcoef(s_main, s_2)[0, 1]),
            'main_vs_s3': float(np.corrcoef(s_main, s_3)[0, 1]),
            's2_vs_s3': float(np.corrcoef(s_2, s_3)[0, 1]),
        }
    }

In [13]:
# All calval
s_main_all = calval['if_score_raw_main'].to_numpy()
s_2_all = calval['if_score_raw_s2'].to_numpy()
s_3_all = calval['if_score_raw_s3'].to_numpy()
stab_all = compute_stability_metrics(s_main_all, s_2_all, s_3_all)

# Clean calval only
clean = calval[calval['is_clean_reference'] == 1]
s_main_clean = clean['if_score_raw_main'].to_numpy()
s_2_clean = clean['if_score_raw_s2'].to_numpy()
s_3_clean = clean['if_score_raw_s3'].to_numpy()
stab_clean = compute_stability_metrics(s_main_clean, s_2_clean, s_3_clean)

del calval, clean
gc.collect()

stability = {
    'all_calibration_val': stab_all,
    'clean_calibration_val': stab_clean,
}

In [14]:
print(f'\n=== Stability on ALL calibration_val ===')
print(f'Jaccard top-1% (n_top = {int(n_all * 0.01):,}):')
for k, v in stab_all['jaccard_top_1pct'].items():
    print(f'  {k}: {v:.4f}')
print(f'Jaccard top-0.5%:')
for k, v in stab_all['jaccard_top_0_5pct'].items():
    print(f'  {k}: {v:.4f}')
print(f'Jaccard top-0.1%:')
for k, v in stab_all['jaccard_top_0_1pct'].items():
    print(f'  {k}: {v:.4f}')
print(f'Pearson:')
for k, v in stab_all['pearson'].items():
    print(f'  {k}: {v:.4f}')

print(f'\n=== Stability on CLEAN calibration_val ===')
print(f'Jaccard top-1% (n_top = {int(n_clean * 0.01):,}):')
for k, v in stab_clean['jaccard_top_1pct'].items():
    print(f'  {k}: {v:.4f}')
print(f'Jaccard top-0.5%:')
for k, v in stab_clean['jaccard_top_0_5pct'].items():
    print(f'  {k}: {v:.4f}')
print(f'Jaccard top-0.1%:')
for k, v in stab_clean['jaccard_top_0_1pct'].items():
    print(f'  {k}: {v:.4f}')
print(f'Pearson:')
for k, v in stab_clean['pearson'].items():
    print(f'  {k}: {v:.4f}')


=== Stability on ALL calibration_val ===
Jaccard top-1% (n_top = 213,902):
  main_vs_s2: 0.7042
  main_vs_s3: 0.7500
  s2_vs_s3: 0.7376
Jaccard top-0.5%:
  main_vs_s2: 0.6891
  main_vs_s3: 0.7218
  s2_vs_s3: 0.6987
Jaccard top-0.1%:
  main_vs_s2: 0.6033
  main_vs_s3: 0.6241
  s2_vs_s3: 0.5810
Pearson:
  main_vs_s2: 0.9879
  main_vs_s3: 0.9864
  s2_vs_s3: 0.9872

=== Stability on CLEAN calibration_val ===
Jaccard top-1% (n_top = 210,835):
  main_vs_s2: 0.7010
  main_vs_s3: 0.7479
  s2_vs_s3: 0.7333
Jaccard top-0.5%:
  main_vs_s2: 0.6806
  main_vs_s3: 0.7153
  s2_vs_s3: 0.6946
Jaccard top-0.1%:
  main_vs_s2: 0.5963
  main_vs_s3: 0.6206
  s2_vs_s3: 0.5645
Pearson:
  main_vs_s2: 0.9879
  main_vs_s3: 0.9864
  s2_vs_s3: 0.9871


In [15]:
# Interpretation
mean_jaccard_all_1pct = np.mean(list(stab_all['jaccard_top_1pct'].values()))
mean_jaccard_clean_1pct = np.mean(list(stab_clean['jaccard_top_1pct'].values()))

print(f'\n=== Interpretation ===')
print(f'Mean Jaccard top-1% on all:   {mean_jaccard_all_1pct:.4f}')
print(f'Mean Jaccard top-1% on clean: {mean_jaccard_clean_1pct:.4f}')

if mean_jaccard_clean_1pct > 0.7:
    print('  Clean stability: STABLE → модель воспроизводима на independent samples.')
elif mean_jaccard_clean_1pct > 0.5:
    print('  Clean stability: MODERATE — приемлемо, document as limitation.')
else:
    print('  Clean stability: UNSTABLE — рассмотреть n_estimators / max_samples↑.')

# Save stability
with open(IF_STABILITY_PATH, 'w') as f:
    json.dump(stability, f, indent=2)
print(f'\nSaved: {IF_STABILITY_PATH}')


=== Interpretation ===
Mean Jaccard top-1% on all:   0.7306
Mean Jaccard top-1% on clean: 0.7274
  Clean stability: STABLE → модель воспроизводима на independent samples.

Saved: /content/drive/MyDrive/thesis_processed/models_v3_artifacts/if_stability.json


## 9. Top anomalies preview (sanity check)

Печатаем top-10 anomalies, отдельно для clean и DQ-tainted, чтобы
убедиться, что IF выявляет что-то осмысленное и DQ-аware
интерпретация работает.

In [16]:
print('\nTop-10 anomalies preview (sanity check)...')

top_data = pq.read_table(
    IF_SCORES_PATH,
    columns=['flight_id', 'split', 'flight_phase',
             'if_score_raw_main', 'if_score_global_pct', 'if_score_phase_pct',
             'dq_hard_flag', 'dq_soft_flag', 'feature_quality_flag',
             'is_clean_reference']
).to_pandas()
top_data['phase_name'] = top_data['flight_phase'].map(
    lambda p: PHASE_NAMES.get(p, 'unknown')
)

# Top-10 overall
print('\n--- Top-10 overall by if_score_raw_main ---')
print(f'{"flight_id":>12s} | {"split":>16s} | {"phase":>10s} | '
      f'{"raw":>8s} | {"glob%":>7s} | {"phase%":>7s} | {"hard":>4s} | {"soft":>4s} | {"fq":>3s}')
print('-' * 110)
for _, row in top_data.nlargest(10, 'if_score_raw_main').iterrows():
    print(f'{int(row["flight_id"]):>12d} | {row["split"]:>16s} | '
          f'{row["phase_name"]:>10s} | {row["if_score_raw_main"]:>8.4f} | '
          f'{row["if_score_global_pct"]:>7.4f} | {row["if_score_phase_pct"]:>7.4f} | '
          f'{int(row["dq_hard_flag"]):>4d} | {int(row["dq_soft_flag"]):>4d} | '
          f'{int(row["feature_quality_flag"]):>3d}')

# Top-10 clean
clean_top = top_data[top_data['is_clean_reference'] == 1].nlargest(10, 'if_score_raw_main')
print('\n--- Top-10 CLEAN (potential operational anomalies) ---')
print(f'{"flight_id":>12s} | {"split":>16s} | {"phase":>10s} | '
      f'{"raw":>8s} | {"glob%":>7s} | {"phase%":>7s}')
print('-' * 80)
for _, row in clean_top.iterrows():
    print(f'{int(row["flight_id"]):>12d} | {row["split"]:>16s} | '
          f'{row["phase_name"]:>10s} | {row["if_score_raw_main"]:>8.4f} | '
          f'{row["if_score_global_pct"]:>7.4f} | {row["if_score_phase_pct"]:>7.4f}')

# Top-10 DQ-tainted
tainted = top_data[
    (top_data['dq_hard_flag'] == 1) | (top_data['feature_quality_flag'] == 1)
].nlargest(10, 'if_score_raw_main')
print('\n--- Top-10 DQ-TAINTED (likely artifacts) ---')
print(f'{"flight_id":>12s} | {"phase":>10s} | {"raw":>8s} | '
      f'{"hard":>4s} | {"soft":>4s} | {"fq":>3s}')
print('-' * 65)
for _, row in tainted.iterrows():
    print(f'{int(row["flight_id"]):>12d} | {row["phase_name"]:>10s} | '
          f'{row["if_score_raw_main"]:>8.4f} | '
          f'{int(row["dq_hard_flag"]):>4d} | {int(row["dq_soft_flag"]):>4d} | '
          f'{int(row["feature_quality_flag"]):>3d}')

del top_data, clean_top, tainted
gc.collect()


Top-10 anomalies preview (sanity check)...

--- Top-10 overall by if_score_raw_main ---
   flight_id |            split |      phase |      raw |   glob% |  phase% | hard | soft |  fq
--------------------------------------------------------------------------------------------------------------
   256925014 |             test |     cruise |   0.7227 |  1.0000 |  1.0000 |    0 |    0 |   0
   256962930 |             test |     cruise |   0.7220 |  1.0000 |  1.0000 |    1 |    0 |   0
   256962930 |             test |     cruise |   0.7220 |  1.0000 |  1.0000 |    1 |    0 |   0
   256962930 |             test |     cruise |   0.7220 |  1.0000 |  1.0000 |    1 |    0 |   0
   256962930 |             test |     cruise |   0.7220 |  1.0000 |  1.0000 |    1 |    0 |   0
   256962930 |             test |     cruise |   0.7220 |  1.0000 |  1.0000 |    1 |    0 |   0
   256805431 |  calibration_val |      climb |   0.7162 |  1.0000 |  1.0000 |    1 |    0 |   0
   256925318 |             test 

0

## 10. Summary

In [17]:
print('\n' + '=' * 60)
print('03.2 SUMMARY')
print('=' * 60)
print(f'\nTraining sample (phase-balanced, clean train_core):')
print(f'  Total: {len(phase_train):,} points')

print(f'\nIF parameters:')
print(f'  n_estimators: {IF_N_ESTIMATORS}, max_samples: {IF_MAX_SAMPLES}')
print(f'  3 random seeds: {IF_SEED_MAIN}, {IF_SEED_2}, {IF_SEED_3}')

print(f'\nScored points: {total_written:,}')
print(f'Phase pct fallback to global: {n_phase_fallback:,}')

print(f'\nStability (mean Jaccard top-1%):')
print(f'  All calibration_val:   {mean_jaccard_all_1pct:.4f}')
print(f'  Clean calibration_val: {mean_jaccard_clean_1pct:.4f}')

print(f'\nArtifacts:')
for path in [IF_MAIN_PATH, IF_SEED2_PATH, IF_SEED3_PATH,
             IF_SCORES_PATH, IF_STABILITY_PATH]:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f'  ✓ {os.path.basename(path):<30s} ({size_mb:>7.1f} MB)')
    else:
        print(f'  ✗ MISSING: {os.path.basename(path)}')

print('\n03.2 complete. Ready for 03.3 (HDBSCAN per phase group).')


03.2 SUMMARY

Training sample (phase-balanced, clean train_core):
  Total: 700,000 points

IF parameters:
  n_estimators: 300, max_samples: 2048
  3 random seeds: 1321, 2321, 3321

Scored points: 62,578,761
Phase pct fallback to global: 0

Stability (mean Jaccard top-1%):
  All calibration_val:   0.7306
  Clean calibration_val: 0.7274

Artifacts:
  ✓ if_main.joblib                 (   12.9 MB)
  ✓ if_seed_2.joblib               (   13.1 MB)
  ✓ if_seed_3.joblib               (   12.7 MB)
  ✓ if_scores.parquet              ( 1363.4 MB)
  ✓ if_stability.json              (    0.0 MB)

03.2 complete. Ready for 03.3 (HDBSCAN per phase group).
